# RAGtune — LLM Agent vs Bayesian TPE Optimizer Benchmark

End-to-end comparison of two hyperparameter optimization strategies for RAGtune pipelines across multiple BEIR benchmarks.

| Method | Strategy |
|--------|----------|
| **Bayesian TPE** | Optuna multi-objective TPE with Pareto pruning (syftr-style) |
| **LLM Agent** | Trace-reflection loop: agent observes execution diagnostics, proposes next config (GEPA/TextGrad-style) |

Both optimizers share the same search space and BM25 retriever (BEIR-tuned k1=0.9, b=0.4).

**Datasets:** TREC-COVID · NFCorpus · SciFact · FIQA  
**Runtime estimate:** ~3–4 hours on Colab T4 for all 4 datasets at 50 iterations each.

## 1 · Environment Setup
Run once per session. Clones the repo, installs dependencies, verifies Java.

In [ ]:
import os, sys

# ── Install system deps ───────────────────────────────────────────────────────
!apt-get install -y -q default-jdk 2>/dev/null
!pip install -q python-terrier ir-datasets litellm optuna matplotlib

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_DIR = "/content/ragtune_repo"
BRANCH   = "feat/llm-optimizer"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/avishekanand/sir.git {REPO_DIR} --quiet

os.chdir(REPO_DIR)
!git checkout {BRANCH} --quiet
!git pull origin {BRANCH} --quiet
!pip install -q -e ".[tuning]"

# Ensure src/ is on sys.path (guards against pip -e not landing in kernel path)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# ── Verify ────────────────────────────────────────────────────────────────────
import subprocess
java = subprocess.run(['java', '-version'], capture_output=True, text=True)
print("Java  :", java.stderr.split('\n')[0])
print("Dir   :", os.getcwd())
print("Python:", sys.version.split()[0])

## 2 · API Key
The LLM optimizer calls `gpt-4o-mini` via litellm. Store your key in Colab Secrets (`🔑` sidebar) as `OPENAI_API_KEY`.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"
    print("Using placeholder key — update before running LLM optimizer.")

## 3 · PyTerrier Init

In [ ]:
import pyterrier as pt

if not pt.started():
    pt.init()

print("PyTerrier", pt.__version__)

## 4 · Dataset Catalogue & Benchmark Settings

Comment out any datasets you don't want to run.  
Adjust `N_EVAL_QUERIES` and `BUDGET` to trade speed for reliability.

| Setting | Value | Notes |
|---------|-------|-------|
| `N_EVAL_QUERIES` | 50 | Full TREC-COVID set; representative sample for others |
| `BUDGET` | 50 | Iterations (LLM) = Trials (Bayes) for fair comparison |
| BM25 k1 / b | 0.9 / 0.4 | Anserini BEIR defaults |

**Quick smoke-test:** set `N_EVAL_QUERIES=5`, `BUDGET=10`, comment out all but `nfcorpus` (~10 min).

In [ ]:
# ── Datasets ──────────────────────────────────────────────────────────────────
DATASETS = {
    "trec-covid": {
        "irds_id":    "irds:beir/trec-covid",
        "index_path": "./idx_trec_covid",
        "n_queries":  50,   # full set — every TREC-COVID topic
    },
    "nfcorpus": {
        "irds_id":    "irds:beir/nfcorpus/test",
        "index_path": "./idx_nfcorpus",
        "n_queries":  50,
    },
    "scifact": {
        "irds_id":    "irds:beir/scifact/test",
        "index_path": "./idx_scifact",
        "n_queries":  50,
    },
    "fiqa": {
        "irds_id":    "irds:beir/fiqa/test",
        "index_path": "./idx_fiqa",
        "n_queries":  50,
    },
}

# ── Optimizer settings ────────────────────────────────────────────────────────
N_EVAL_QUERIES = 50
BUDGET         = 50
SEED           = 42
LLM_MODEL      = "gpt-4o-mini"
BM25_K1        = 0.9
BM25_B         = 0.4

# Identical search space for both optimizers
SEARCH_SPACE = {
    "reranker_types":     ["noop", "cross-encoder", "monot5"],
    "reformulator_types": ["identity"],
    "estimator_types":    ["baseline", "utility", "similarity"],
    "scheduler_types":    ["active-learning", "graceful-degradation"],
    "feedback_types":     ["none", "budget-stop"],
}

print(f"Datasets  : {list(DATASETS.keys())}")
print(f"Budget    : {BUDGET} evals × {N_EVAL_QUERIES} queries = "
      f"{BUDGET * N_EVAL_QUERIES} total query runs per optimizer per dataset")
print(f"BM25      : k1={BM25_K1}, b={BM25_B}  (BEIR Anserini defaults)")
print(f"LLM model : {LLM_MODEL}")

## 5 · Build Indexes
Skips any dataset whose index already exists on disk.  
**Index build times (Colab T4):** trec-covid ~5 min · nfcorpus ~30 s · scifact ~45 s · fiqa ~3 min

In [ ]:
from pathlib import Path

def build_index(irds_id: str, index_path: str) -> None:
    if Path(index_path + "/data.properties").exists():
        n = pt.IndexFactory.of(index_path).getCollectionStatistics().getNumberOfDocuments()
        print(f"  [{index_path}] already built ({n:,} docs) — skipping")
        return
    print(f"  [{index_path}] indexing {irds_id} …")
    ds = pt.get_dataset(irds_id)
    indexer = pt.IterDictIndexer(
        index_path,
        overwrite=True,
        meta={"docno": 26, "text": 131072},
        text_attrs=["text"],
    )
    indexer.index(ds.get_corpus_iter())
    n = pt.IndexFactory.of(index_path).getCollectionStatistics().getNumberOfDocuments()
    print(f"  [{index_path}] done — {n:,} documents")

print("Building indexes …")
for name, cfg in DATASETS.items():
    print(f"\n{name}:")
    build_index(cfg["irds_id"], cfg["index_path"])

print("\nAll indexes ready.")

## 6 · BM25 Sanity Check
Confirm the BEIR-tuned BM25 parameters give expected NDCG@10 before running the full optimization loop.

In [ ]:
import ragtune.adapters, ragtune.components  # populate registry
from ragtune.adapters.pyterrier import PyTerrierRetriever
from ragtune.tuning.evaluator import EvalDataset, ndcg_at_k
from ragtune.core.types import RAGtuneContext
from ragtune.core.budget import CostTracker, CostBudget
from ragtune.core.controller import ControllerTrace
import numpy as np

# Published BEIR BM25 baselines (Anserini, k1=0.9 b=0.4) for reference
BEIR_BM25_REF = {
    "trec-covid": 0.656,
    "nfcorpus":   0.325,
    "scifact":    0.665,
    "fiqa":       0.236,
}

print(f"{'Dataset':<14} {'Our BM25':>10} {'BEIR ref':>10} {'Gap':>8}")
print("-" * 46)

for ds_name, cfg in DATASETS.items():
    br = pt.BatchRetrieve(
        cfg["index_path"],
        wmodel="BM25",
        controls={"BM25.b": str(BM25_B), "BM25.k_1": str(BM25_K1)},
        metadata=["docno", "text"],
        num_results=1000,
    )
    dataset = EvalDataset.from_pyterrier_irds(
        irds_id=cfg["irds_id"], n_queries=cfg["n_queries"], seed=SEED
    )
    scores = []
    for eq in dataset.iter_queries(limit=cfg["n_queries"]):
        res = br.search(eq.query)
        ranked_ids = list(res["docno"].astype(str))
        scores.append(ndcg_at_k(ranked_ids, eq.qrels, k=10))
    our_ndcg = float(np.mean(scores))
    ref      = BEIR_BM25_REF.get(ds_name, float("nan"))
    gap      = our_ndcg - ref
    print(f"{ds_name:<14} {our_ndcg:>10.4f} {ref:>10.3f} {gap:>+8.3f}")

print("\nGap within ±0.05 of BEIR reference is expected (different index / tokeniser).")

## 7 · Run Both Optimizers on Every Dataset

This is the main optimization loop. For each dataset:
1. Builds a shared BM25 retriever (k1=0.9, b=0.4)
2. Runs the **LLM agent** optimizer for `BUDGET` iterations
3. Runs the **Bayesian TPE** optimizer for `BUDGET` trials

Results are stored in `all_results` for plotting in the next cell.  
Pareto-optimal configs are written to `./results_<dataset>_{llm,bayes}/`.

In [ ]:
import time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from ragtune.tuning.study_config import TuningStudyConfig, DatasetConfig
from ragtune.tuning.optimizer import run_study, extract_pareto_configs
from ragtune.tuning.search_space import RAGtuneSearchSpace
from ragtune.tuning.llm_optimizer import (
    LLMOptimizerConfig, LLMAgentOptimizer, compute_pareto_front,
)

all_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'='*65}")
    print(f"  {ds_name.upper()}")
    print(f"{'='*65}")

    # ── Shared retriever (BEIR-tuned BM25) ───────────────────────────────────
    br = pt.BatchRetrieve(
        ds_cfg["index_path"],
        wmodel="BM25",
        controls={"BM25.b": str(BM25_B), "BM25.k_1": str(BM25_K1)},
        metadata=["docno", "text"],
        num_results=1000,
    )
    fixed_retriever = PyTerrierRetriever(pt_transformer=br)

    # ── Queries + qrels ───────────────────────────────────────────────────────
    full_dataset = EvalDataset.from_pyterrier_irds(
        irds_id=ds_cfg["irds_id"],
        n_queries=ds_cfg["n_queries"],
        seed=SEED,
    )
    print(f"  Loaded {len(full_dataset.queries)} queries")

    # ── LLM agent ─────────────────────────────────────────────────────────────
    print(f"  [LLM] Running {BUDGET} iterations …")
    llm_cfg = LLMOptimizerConfig(
        name=f"llm-{ds_name}",
        llm_model=LLM_MODEL,
        temperature=0.7,
        n_iterations=BUDGET,
        n_eval_queries=N_EVAL_QUERIES,
        seed=SEED,
        output_dir=f"./results_{ds_name}_llm",
        search_space_overrides=SEARCH_SPACE,
    )
    llm_opt = LLMAgentOptimizer(config=llm_cfg)
    t0 = time.time()
    llm_history = llm_opt.run(fixed_retriever, full_dataset)
    llm_time    = time.time() - t0
    llm_pareto  = compute_pareto_front(llm_history)
    llm_valid   = [c for c in llm_history if not c.error]
    print(f"  [LLM] Done in {llm_time:.0f}s — "
          f"{len(llm_valid)} ok, Pareto size {len(llm_pareto)}")

    # ── Bayesian TPE ──────────────────────────────────────────────────────────
    print(f"  [Bayes] Running {BUDGET} trials …")
    bayes_cfg = TuningStudyConfig(
        name=f"bayes-{ds_name}",
        dataset=DatasetConfig(name=ds_name, irds_id=ds_cfg["irds_id"]),
        n_trials=BUDGET,
        n_startup_trials=max(5, BUDGET // 8),
        n_eval_queries=N_EVAL_QUERIES,
        seed=SEED,
        n_parallel_workers=1,
        max_mean_rerank_docs=200.0,
        max_trial_seconds=300.0,
        pareto_warmup_trials=max(5, BUDGET // 8),
        output_dir=f"./results_{ds_name}_bayes",
        search_space_overrides=SEARCH_SPACE,
    )
    t0 = time.time()
    study = run_study(bayes_cfg, fixed_retriever, full_dataset)
    bayes_time     = time.time() - t0
    bayes_complete = [t for t in study.trials if t.values is not None]
    print(f"  [Bayes] Done in {bayes_time:.0f}s — "
          f"{len(bayes_complete)} ok, Pareto size {len(study.best_trials)}")

    # ── Save ──────────────────────────────────────────────────────────────────
    all_results[ds_name] = {
        "llm_history":    llm_history,
        "llm_valid":      llm_valid,
        "llm_pareto":     llm_pareto,
        "llm_time":       llm_time,
        "study":          study,
        "bayes_complete": bayes_complete,
        "bayes_time":     bayes_time,
    }

print("\n\nAll datasets complete.")

## 8 · Results Table

Best NDCG@10 and hypervolume (area under the Pareto frontier — higher = better on both axes).

In [ ]:
import numpy as np

REF_COST = 200.0  # upper bound on cost axis for hypervolume calculation

def hypervolume_2d(pairs, ref_cost=REF_COST):
    """Dominated hypervolume (maximize NDCG, minimize cost)."""
    pts = sorted([(n, c) for n, c in pairs if c < ref_cost and n > 0], key=lambda p: p[1])
    hv, prev = 0.0, 0.0
    for ndcg, cost in pts:
        hv += ndcg * (cost - prev)
        prev = cost
    return hv

rows = []
for ds_name, res in all_results.items():
    llm_ndcg   = max((c.ndcg_at_10 for c in res["llm_valid"]), default=0.0)
    bayes_ndcg = max((t.values[0] for t in res["bayes_complete"] if t.values), default=0.0)
    llm_hv = hypervolume_2d(
        [(c.ndcg_at_10, c.mean_rerank_docs) for c in res["llm_pareto"]]
    )
    bayes_hv = hypervolume_2d(
        [(t.values[0], t.values[1]) for t in res["study"].best_trials if t.values]
    )
    rows.append({
        "dataset":      ds_name,
        "llm_ndcg":     llm_ndcg,
        "bayes_ndcg":   bayes_ndcg,
        "llm_hv":       llm_hv,
        "bayes_hv":     bayes_hv,
        "bayes_pareto": len(res["study"].best_trials),
        "llm_pareto":   len(res["llm_pareto"]),
        "llm_time":     res["llm_time"],
        "bayes_time":   res["bayes_time"],
    })

W = 78
print("=" * W)
print(f"  Multi-benchmark results  |  "
      f"{BUDGET} evals × {N_EVAL_QUERIES} queries  |  BM25 k1={BM25_K1} b={BM25_B}")
print("=" * W)
print(f"  {'Dataset':<13}  {'LLM NDCG':>9}  {'Bayes NDCG':>10}  "
      f"{'LLM HV':>8}  {'Bayes HV':>8}  {'Pareto':>6}")
print("-" * W)
for r in rows:
    ndcg_w = " ◀" if r["llm_ndcg"] > r["bayes_ndcg"] else "  "
    hv_w   = " ◀" if r["llm_hv"]   > r["bayes_hv"]   else "  "
    print(
        f"  {r['dataset']:<13}  "
        f"{r['llm_ndcg']:>9.4f}{ndcg_w} "
        f"{r['bayes_ndcg']:>10.4f}  "
        f"{r['llm_hv']:>8.2f}{hv_w} "
        f"{r['bayes_hv']:>8.2f}  "
        f"{r['llm_pareto']:>2} / {r['bayes_pareto']:<2}"
    )
print("=" * W)
print("  ◀ = better on this metric   Pareto = LLM / Bayes front size")

print("\n  Wall time (s):")
for r in rows:
    print(f"    {r['dataset']:<13}  LLM {r['llm_time']:>6.0f}s   Bayes {r['bayes_time']:>6.0f}s")

## 9 · Visualization

One row per dataset: **Pareto scatter** (all evaluated points + frontier step) and **convergence curve** (best NDCG found so far).

In [ ]:
import matplotlib.pyplot as plt

n_ds = len(all_results)
fig, axes = plt.subplots(n_ds, 2, figsize=(14, 4.5 * n_ds))
if n_ds == 1:
    axes = [axes]

LLM_CLR   = "#1565C0"
BAYES_CLR = "#B71C1C"

def _best_so_far(values):
    best, out = 0.0, []
    for v in values:
        if v is not None:
            best = max(best, v)
        out.append(best)
    return out

def _pareto_step(points_xy, ax, color, label):
    if not points_xy:
        return
    pts = sorted(points_xy, key=lambda p: p[0])
    xs, ys = zip(*pts)
    ax.step(xs, ys, where="pre", color=color, linewidth=2.2, alpha=0.85)
    ax.scatter(xs, ys, color=color, s=90, zorder=6, label=label,
               edgecolors="white", linewidths=0.8)

for row_i, (ds_name, res) in enumerate(all_results.items()):
    ax_p, ax_c = axes[row_i]
    study          = res["study"]
    bayes_complete = res["bayes_complete"]
    llm_valid      = res["llm_valid"]
    llm_pareto     = res["llm_pareto"]

    # ── Pareto scatter ────────────────────────────────────────────────────────
    ax_p.scatter(
        [c.mean_rerank_docs for c in llm_valid],
        [c.ndcg_at_10 for c in llm_valid],
        alpha=0.2, color=LLM_CLR, s=22, label="LLM — all evals",
    )
    ax_p.scatter(
        [t.values[1] for t in bayes_complete],
        [t.values[0] for t in bayes_complete],
        alpha=0.2, color=BAYES_CLR, s=22, marker="s", label="Bayes — all evals",
    )
    _pareto_step(
        [(c.mean_rerank_docs, c.ndcg_at_10) for c in llm_pareto],
        ax_p, LLM_CLR, "LLM — Pareto",
    )
    _pareto_step(
        [(t.values[1], t.values[0]) for t in study.best_trials if t.values],
        ax_p, BAYES_CLR, "Bayes — Pareto",
    )
    ax_p.set_xlabel("Mean rerank docs consumed (cost ↓)", fontsize=10)
    ax_p.set_ylabel("NDCG@10 (quality ↑)", fontsize=10)
    ax_p.set_title(f"{ds_name} — Pareto Front", fontsize=11, fontweight="bold")
    ax_p.legend(fontsize=8)
    ax_p.grid(True, alpha=0.3)

    # ── Convergence ───────────────────────────────────────────────────────────
    llm_curve = _best_so_far(
        [c.ndcg_at_10 if not c.error else None for c in res["llm_history"]]
    )
    bayes_curve = _best_so_far([
        t.values[0] if t.values else None
        for t in sorted(study.trials, key=lambda x: x.number)
    ])
    iters = range(1, BUDGET + 1)
    ax_c.plot(iters, llm_curve,   color=LLM_CLR,   linewidth=2.2,
              marker="o", markersize=4, label="LLM agent")
    ax_c.plot(iters, bayes_curve, color=BAYES_CLR, linewidth=2.2,
              marker="s", markersize=4, linestyle="--", label="Bayesian TPE")
    ax_c.set_xlabel("Iteration / Trial", fontsize=10)
    ax_c.set_ylabel("Best NDCG@10 so far", fontsize=10)
    ax_c.set_title(f"{ds_name} — Convergence", fontsize=11, fontweight="bold")
    ax_c.legend(fontsize=9)
    ax_c.grid(True, alpha=0.3)
    ax_c.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("ragtune_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ragtune_benchmark.png")

## 10 · LLM Agent Reasoning Trace
Inspect what the agent was thinking across iterations for a chosen dataset.

In [ ]:
SHOW_DATASET = "trec-covid"   # change to any key in DATASETS

res = all_results[SHOW_DATASET]
print(f"=== LLM Agent Reasoning Trace — {SHOW_DATASET} ===\n")
for c in res["llm_history"]:
    if c.error:
        print(f"Iter {c.iteration:2d}  [ERROR] {c.error[:80]}")
        continue
    t = c.trace
    star = " ★" if any(p is c for p in res["llm_pareto"]) else ""
    print(
        f"Iter {c.iteration:2d}{star}  "
        f"NDCG={c.ndcg_at_10:.4f}  cost={c.mean_rerank_docs:5.1f}  "
        f"reranker={c.params.get('reranker_type','?'):<14}  "
        f"depth={c.params.get('original_query_depth','?')}  "
        f"pool={t.avg_pool_size:.0f}  reranked={t.avg_pct_pool_reranked:.0%}"
    )
    print(f"         {(c.rationale or '')[:200]}")
    print()

## 11 · Pareto Config Inspection
Print the winning pipeline configurations from both optimizers.

In [ ]:
import yaml
from pathlib import Path

for ds_name in all_results:
    for label, out_dir in [("LLM", f"./results_{ds_name}_llm"),
                           ("Bayes", f"./results_{ds_name}_bayes")]:
        files = sorted(Path(out_dir).glob("*.yaml")) if Path(out_dir).exists() else []
        if not files:
            continue
        print(f"\n{'='*60}")
        print(f"  {label} Pareto — {ds_name} ({len(files)} configs)")
        print(f"{'='*60}")
        for f in files:
            cfg   = yaml.safe_load(f.read_text())
            comps  = cfg["pipeline"]["components"]
            budget = cfg["pipeline"]["budget"]["limits"]
            print(f"  {f.name}")
            print(f"    reranker  : {comps['reranker']['type']}")
            print(f"    scheduler : {comps['scheduler']['type']}")
            print(f"    estimator : {comps['estimator']['type']}")
            print(f"    rerank_budget: {budget['rerank_docs']}")
            print()

## 12 · Export Results to CSV
Flat CSV of every evaluated configuration, useful for further analysis.

In [ ]:
import csv, io

rows_csv = []

for ds_name, res in all_results.items():
    for c in res["llm_history"]:
        if c.error:
            continue
        row = {"dataset": ds_name, "method": "llm",
               "iteration": c.iteration, "ndcg_at_10": c.ndcg_at_10,
               "mean_rerank_docs": c.mean_rerank_docs}
        row.update(c.params)
        rows_csv.append(row)

    for t in res["bayes_complete"]:
        if t.values is None:
            continue
        row = {"dataset": ds_name, "method": "bayes",
               "iteration": t.number, "ndcg_at_10": t.values[0],
               "mean_rerank_docs": t.values[1]}
        row.update(t.params)
        rows_csv.append(row)

if rows_csv:
    fieldnames = list(rows_csv[0].keys())
    with open("ragtune_results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows_csv)
    print(f"Saved ragtune_results.csv — {len(rows_csv)} rows")
    print(f"Columns: {fieldnames[:8]} …")
else:
    print("No results to export.")